# Import Essential Libraries

In [2]:
# Essential Libraries Import
import cv2
import numpy as np
import matplotlib.pyplot as plt

# final road crack detection system

In [12]:
# load video
input_path = "in.mov"
cap = cv2.VideoCapture(input_path)

if not cap.isOpened():
    print("Error: Video file eka open karanna ba.")
else:
    # get frame details (Frame width, height, FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Output 1: Enhanced Grayscale Video (B&W)
    out_enhanced = cv2.VideoWriter('enhanced_output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height), isColor=False)
    # Output 2: Final Detection Video (Color)
    out_final = cv2.VideoWriter('crack_detection_final.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height), isColor=True)

    
    print("Processing started... press 'q' to stop...")

    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # convert image to gray scale to reduce the computation power and time
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
        # Power-Law (Gamma) Transformation
        # Used to increase contrast by darkening pixels.
        # This makes road cracks appear much darker and easier to detect.
        gamma = 2.0
        gamma_corrected = np.array(255 * (gray / 255) ** gamma, dtype='uint8')
    
        # Median Filtering (Noise Reduction)
        # Used to remove small dots, sand, and salt-and-pepper noise.
        # This smooths the road surface while preserving the sharp edges of the cracks.
        median_filtered = cv2.medianBlur(gamma_corrected, 5)
    
        # Laplacian Sharpening
        # Used to detect intensity changes and highlight the edges of the cracks.
        # It makes the crack boundaries appear as bright white lines against the background.
        laplacian = cv2.Laplacian(median_filtered, cv2.CV_64F)
        sharpened_edges = np.uint8(np.absolute(laplacian))
    
        # Min-Max Normalization (Contrast Stretching)
        # Used to expand the pixel range to the full 0-255 scale.
        # This makes the faint crack edges much brighter and more distinct.
        stretched = cv2.normalize(sharpened_edges, None, 0, 255, cv2.NORM_MINMAX)
    
        # Image Subtraction
        # Used to darken the crack areas by subtracting the bright edges from the filtered image.
        # Since the cracks are now represented by white pixels (high values) in the 'stretched' image,
        # subtracting them from the original makes those specific areas turn black.
        final_enhanced = cv2.subtract(median_filtered, stretched)
    
        # Final Normalization (Optional)
        # Used to stretch the pixel intensity to the full 0-255 range.
        # This maximizes the contrast, making the background brighter and the cracks darker.
        final_enhanced = cv2.normalize(final_enhanced, None, 0, 255, cv2.NORM_MINMAX)
    
        # save the processed frame
        out_enhanced.write(final_enhanced)


        # Statistical Analysis
        # Calculating the brightness (Mean) and contrast (Standard Deviation) 
        # to understand how noisy or clear the image is.
        mean_val = np.mean(final_enhanced)
        std_val = np.std(final_enhanced)
        
        # Coefficient of Variation (Ratio):
        # This helps decide if we need heavy cleaning (Opening) or just connecting lines (Closing) based on the image's texture.
        ratio = std_val / mean_val if mean_val != 0 else 0
        
        # Binary Thresholding
        # Separating the crack pixels from the road. 
        # This creates a clear mask where the defects are isolated from the background.
        _, crack_mask = cv2.threshold(final_enhanced, 35, 255, cv2.THRESH_BINARY_INV)
        
        
        # We classify images into 3 categories based on their Variation Ratio to apply the most suitable cleaning and connecting filters.
        
        if ratio > 0.6: 
            # Extremely High Contrast (Clear Images)
            # The crack is already prominent, so we avoid 'Opening' to prevent losing detail.
            # For visualization consistency, we map 'cleaned' to the original mask.
            cleaned = crack_mask.copy() 
            
            # Using a small 3x3 kernel for a gentle 'Closing' to bridge tiny gaps.
            kernel_cl = np.ones((3,3), np.uint8)
            connected = cv2.morphologyEx(crack_mask, cv2.MORPH_CLOSE, kernel_cl, iterations=1)
            
        elif 0.4 < ratio <= 0.6: 
            # Medium Contrast (Mild Noise/Texture)
            # Requires a balance between cleaning and detail preservation. 
            # 'Opening' with a tiny 2x2 kernel to remove micro-noise without erasing fine cracks.
            kernel_small = np.ones((2,2), np.uint8) 
            cleaned = cv2.morphologyEx(crack_mask, cv2.MORPH_OPEN, kernel_small, iterations=1)
            
            # 'Closing' with a 5x5 kernel to strengthen the crack structure.
            kernel_cl = np.ones((3,3), np.uint8)
            connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_cl, iterations=2)
            
        else: 
            # Standard/Low Contrast (Fragmented or Noisy Images)
            # Cracks are likely broken into pixels; needs aggressive cleaning and merging.
            # Standard 3x3 'Opening' to eliminate significant road surface artifacts.
            kernel = np.ones((3,3), np.uint8)
            cleaned = cv2.morphologyEx(crack_mask, cv2.MORPH_OPEN, kernel, iterations=2)
            
            # Strong 9x9 'Closing' to bridge large gaps and merge fragmented red-dots into lines.
            kernel_cl = np.ones((9,9), np.uint8)
            connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_cl, iterations=2)

        # This function scans the binary mask and extracts the actual shapes of the cracks.
        # It turns raw pixel data into a list of mathematical objects (contours) 
        # that we can count, measure, and highlight on the final image.
        contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Creating a duplicate of the original image for visualization.
        # We draw our detections on this 'overlay'.
        overlay = frame.copy()

        # Detection Filter & Visualization
        # Iterating through each detected shape to filter out noise and highlight valid cracks.
        for cnt in contours:
            # Calculating the surface area of the detected shape
            area = cv2.contourArea(cnt)
            
            # Noise is not highlighted
            # This ignores tiny dots or textures that aren't road defects.
            if area > 150:
                # Drawing the detection:
                cv2.drawContours(overlay, [cnt], -1, (0, 0, 255), 2)

        # Save and Show results
        out_final.write(overlay)
        # cv2.imshow('Crack Detection Live', overlay)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out_enhanced.release() # Mekai
    out_final.release()    # Mekai wenna ona
    cv2.destroyAllWindows()
    print("processed video saved as a mp4")

Processing started... press 'q' to stop...
processed video saved as a mp4
